In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
import random
import os

# Set environment variables and threads for reproducibility and performance
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['TORCH_NUM_THREADS'] = '1'  # For PyTorch

torch.set_num_threads(1)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

warnings.filterwarnings("ignore")

# Load the dataset
data = pd.read_excel(# add file path)

# Define feature groups
feature_groups = {
    'Genotype': [
        'rs11225395', 'rs1144393', 'rs650108', 'rs591058', 'rs2252070', 'rs4986938', 'rs1800012', 'rs4789932', 'rs9340799', 'rs970547', 
        'rs1800795', 'rs13946', 'rs12722', 'class1_SNP_risk_score', 'rs7528684', 'rs4919510', 'rs1937810', 'rs6481512', 'rs1249269', 
        'rs12574452', 'rs12429486', 'rs4454832', 'rs2761884', 'rs62051384', 'rs4362400', 'rs2586488', 'rs2277698', 'rs1045485', 
        'rs143383', 'rs17576', 'rs2305948', 'rs1011814', 'rs11154027', 'rs2234693', 'rs1643821', 'rs2010963', 'rs10263021', 'rs149047058', 
        'rs420257', 'rs42517', 'rs42522', 'rs42531', 'rs413826', 'rs2104772', 'rs1330363', 'class12_SNP_risk_score', 'rs3753841', 
        'rs57104447', 'rs1887632', 'rs4654760', 'rs1137101', 'rs2306033', 'rs2277268', 'rs4988321', 'rs11232681', 'rs1718119', 'rs3751143', 
        'rs1544410', 'rs2228570', 'rs4328262', 'rs1021188', 'rs74544784', 'rs78391032', 'rs77569527', 'rs117544024', 'rs912336', 
        'rs3218791', 'rs911263', 'rs2525504', 'rs17756404', 'rs4903399', 'rs10132091', 'rs17583842', 'rs1676303', 'rs11629171', 
        'rs2281518', 'rs2285053', 'rs71404070', 'rs710079', 'rs2858056', 'rs820218', 'rs3018362', 'rs1800470', 'rs1800469', 'rs25487', 
        'rs25489', 'rs2289360', 'rs183364169', 'rs11177', 'rs6617', 'rs3219008', 'rs13107325', 'rs60713544', 'rs145648292', 'rs4244032', 
        'rs12656106', 'rs3045', 'rs187483', 'rs4701616', 'rs144414988', 'rs1800629', 'rs10484958', 'rs4730153', 'rs1800797', 'rs1554606', 
        'rs2237352', 'rs4725069', 'rs12154667', 'rs1548456', 'rs3216902', 'rs35360670', 'rs13317', 'rs1800972', 'rs7035322', 'rs7021589', 
        'rs72758637', 'rs10759753', 'rs3789870', 'rs1138545', 'rs3196378', 'rs1134170', 'rs10992075', 'rs1590', 'rs144371252', 
        'rs761804508', 'class123_SNP_risk_score', 'sex'
    ],
    'History': [
        'Age', 'lower_limb_days_total', 'average_run_hours', 'average_interval_training_frequency', 'EDEQ_total', 'tracking_period_injury',
        'past_stress_injury', 'LEAF-Q', 'Athlete_Score', 'average_run_frequency', 'past_month_injury'
    ],
    'Phenotype': [
        'hip_abduction_peak_torque', 'total_ad_ab_ratio', 'knee_extension_peak_torque', 'knee_flexion_peak_torque', 'navicular_drop', 
        'navicular_drop_asymmetry', 'Q_angle', 'Q_angle_asymmetry', 'VALR_12', 'Impact_peak_12', 'Duty_factor_12', 'BMI', 'BMD_spine',
        'hip_abduction_peak_torque_asymmetry', 'hip_adduction_peak_torque', 'hip_adduction_peak_torque_asymmetry', 
        'knee_extension_peak_torque_asymmetry', 'knee_flexion_peak_torque_asymmetry', 'total_fl_ex_ratio', 'leg_lean_mass', 
        'hip_abduction_peak_angle', 'hip_abduction_peak_angle_asymmetry', 'hip_adduction_peak_angle', 'hip_adduction_peak_angle_asymmetry', 
        'ad_ab_ratio_asymmetry', 'knee_extension_peak_angle', 'knee_extension_peak_angle_asymmetry', 'knee_flexion_peak_angle', 
        'knee_flexion_peak_angle_asymmetry', 'fl_ex_ratio_asymmetry', 'VILR_10', 'VALR_10', 'VILR_asymmetry_10', 'VALR_asymmetry_10', 
        'Impact_peak_10', 'Impact_peak_asymmetry_10', 'Flight_time_10', 'Contact_time_10', 'Duty_factor_10', 'Step_frequency_10', 
        'Cadence_asymmetry_10', 'Duty_factor_asymmetry_10', 'VILR_12', 'VILR_asymmetry_12', 'VALR_asymmetry_12', 
        'Impact_peak_asymmetry_12', 'Flight_time_12', 'Contact_time_12', 'Step_frequency_12', 'Cadence_asymmetry_12', 
        'Duty_factor_asymmetry_12', 'Alt_strike', 'height', 'Mass', 'thigh_lean_mass', 'thigh_ffmi', 'lower_leg_lean_mass', 
        'lower_leg_ffmi', 'leg_ffmi', 'total_lean_mass', 'total_ffmi', 'calf_size', 'BMD_hip', 'BMD_body',
    ],
    'Behaviour': [
        'fat_intake_avg', 'past_month_distance', 'past_month_ratio', 'SC_past_season', 'non_running_past_season', 'fat_intake_BW', 
        'fat_percentage_avg', 'average_energy_availability', 'protein_intake_BW', 'omega3_intake_BW', 'vitaminD_intake_BW', 
        'vitaminC_intake_BW', 'vitaminE_intake_BW', 'calcium_intake_BW', 'copper_intake_BW', 'iron_intake_BW', 'glycine_intake_BW', 
        'arginine_intake_BW', 'past_month_min', 'past_week_ratio', 'past_month_volume_low', 'past_week_ratio_low', 'past_month_ratio_low', 
        'past_month_volume_moderate', 'past_week_ratio_moderate', 'past_month_ratio_moderate', 'past_month_volume_high', 
        'past_week_ratio_high', 'past_month_ratio_high', 'past_month_volume_very_high', 'past_week_ratio_very_high', 
        'past_month_ratio_very_high', 'past_month_calculated_volume', 'past_week_ratio_calculated_volume', 
        'past_month_ratio_calculated_volume', 'resistance_training_past_month', 'resistance_training_past_season', 
        'bodyweight_exercises_past_month', 'bodyweight_exercises_past_season', 'core_stability_past_month', 'core_stability_past_season', 
        'balance_training_past_month', 'balance_training_past_season', 'plyometrics_past_month', 'plyometrics_past_season', 
        'drills_past_month', 'drills_past_season', 'circuit_training_past_month', 'circuit_training_past_season', 'barefoot_past_month', 
        'barefoot_past_season', 'stretching_past_month', 'stretching_past_season', 'SC_past_month', 'non_running_past_month'
    ]
}

# Define predictors and outcome
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome

# Ensure that the feature groups exist in the dataset
for group in feature_groups:
    feature_groups[group] = [feature for feature in feature_groups[group] if feature in X.columns]

num_features = len(X.columns)
feature_to_idx = {feature: idx for idx, feature in enumerate(X.columns)}

# Initialize adjacency matrix with zeros
adjacency = np.zeros((num_features, num_features), dtype=np.float32)

# Define groups
genotype_features = feature_groups['Genotype']
history_features = feature_groups['History']
phenotype_features = feature_groups['Phenotype']
behaviour_features = feature_groups['Behaviour']

# Get indices for each group
genotype_indices = [feature_to_idx[feat] for feat in genotype_features]
history_indices = [feature_to_idx[feat] for feat in history_features]
phenotype_indices = [feature_to_idx[feat] for feat in phenotype_features]
behaviour_indices = [feature_to_idx[feat] for feat in behaviour_features]

# Genotype nodes: connect to every node except themselves
for i in genotype_indices:
    for j in range(num_features):
        if j != i:
            adjacency[i, j] = 1.0

# History nodes: connect to every node except genotype nodes and themselves
for i in history_indices:
    for j in range(num_features):
        if j not in genotype_indices and j != i:
            adjacency[i, j] = 1.0

# Phenotype nodes: connect to every node except genotype, history nodes, and themselves
for i in phenotype_indices:
    for j in range(num_features):
        if j not in genotype_indices and j not in history_indices and j != i:
            adjacency[i, j] = 1.0

# Behaviour nodes: connect to all other behaviour nodes except themselves
for i in behaviour_indices:
    for j in behaviour_indices:
        if j != i:
            adjacency[i, j] = 1.0

# Convert adjacency to tensor
adjacency_mask = torch.tensor(adjacency, dtype=torch.float32)

class CustomDataset(Dataset):
    def __init__(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.X = X.values.astype(np.float32)
        elif isinstance(X, np.ndarray):
            self.X = X.astype(np.float32)
        else:
            raise TypeError("X should be a pandas DataFrame or a NumPy array.")
        self.y = y.astype(np.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class CustomNetwork(nn.Module):
    def __init__(self, num_features, adjacency_mask):
        super(CustomNetwork, self).__init__()
        self.num_features = num_features

        # Register the fixed adjacency mask as a buffer (non-trainable)
        self.register_buffer('mask', adjacency_mask.float())  # Shape: (num_features, num_features)

        # Initialize connection weights (W_ij: from node j to node i)
        self.W = nn.Parameter(torch.randn(num_features, num_features) * 0.01)

        # Initialize bias for each node
        self.bias = nn.Parameter(torch.zeros(num_features))

        # Initialize output weights (from each node to the final output)
        self.output_weights = nn.Parameter(torch.randn(num_features) * 0.01)

        # Add Batch Normalization
        self.batch_norm = nn.BatchNorm1d(num_features)

    def forward(self, x, threshold=None):
        """
        x: input tensor of shape (batch_size, num_features)
        threshold: if provided, apply binary thresholding to gating
        """
        # Use the fixed mask directly
        learnable_mask = self.mask

        # Apply the learnable mask to connection weights
        W_masked = self.W * learnable_mask  # Shape: (num_features, num_features)

        # Compute incoming messages: (batch_size, num_features) @ (num_features, num_features) = (batch_size, num_features)
        incoming = torch.matmul(x, W_masked)  # Sum over j for each i: sum_j (x_j * W_ij)

        # Element-wise multiplication with own feature value
        node_input = incoming * x  # Shape: (batch_size, num_features)

        node_input = self.batch_norm(node_input)

        # Add bias
        node_input += self.bias  # Broadcasting over batch

        # Apply Leaky ReLU activation
        node_output = F.leaky_relu(node_input)  # Shape: (batch_size, num_features)

        # Aggregate node outputs to final output
        output = torch.matmul(node_output, self.output_weights)  # Shape: (batch_size,)

        return output  # Raw scores (logits)

# Best hyperparameters from Optuna
best_hyperparams = {
    'n_epochs': 2059,
    'lr': 1.185885489910591e-05,
    'weight_decay': 9.916744984817729e-07,
    'batch_size': 64,
    'n_genotype': 125,
    'n_history': 3,
    'n_phenotype': 44,
    'n_behaviour': 20
}

def global_feature_ranking(X, y):
    """
    Perform global feature ranking using ReliefF.
    """
    from skrebate import ReliefF
    # Initialize ReliefF without random_state
    relief = ReliefF(n_features_to_select='all')

    # Convert X and y to NumPy arrays to avoid KeyError
    X_array = X.values
    y_array = y.values
    
    # Fit ReliefF on the entire dataset
    relief.fit(X_array, y_array)
    
    # Get feature scores
    feature_scores = relief.feature_importances_
    
    # Create a DataFrame for easy handling
    feature_score_df = pd.DataFrame({
        'feature': X.columns,
        'score': feature_scores
    })
    
    # Sort features by score in descending order
    feature_score_df = feature_score_df.sort_values(by='score', ascending=False)
    
    return feature_score_df

# Perform global feature ranking
global_feature_score_df = global_feature_ranking(X, y)

# Select features based on best hyperparameters
selected_features = []
for group, n_select in zip(['Genotype', 'History', 'Phenotype', 'Behaviour'],
                           [best_hyperparams['n_genotype'], best_hyperparams['n_history'],
                            best_hyperparams['n_phenotype'], best_hyperparams['n_behaviour']]):
    group_features = feature_groups[group]
    # Filter the features present in the group
    group_feature_scores = global_feature_score_df[global_feature_score_df['feature'].isin(group_features)]
    # Select top N features
    top_features = group_feature_scores.head(n_select)['feature'].tolist()
    selected_features.extend(top_features)

# Ensure that selected_features are unique
selected_features = list(dict.fromkeys(selected_features))

print(f"Selected Features: {selected_features}")

# Save selected_features for future use
import json
with open("selected_features.json", "w") as f:
    json.dump(selected_features, f)

# Data Preparation
X_selected = X[selected_features].copy()
y_selected = y.copy()

# Encode labels for binary classification (assuming binary classification)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_selected_enc = le.fit_transform(y_selected)

# Create the entire dataset
full_dataset = CustomDataset(X_selected, y_selected_enc)

# Create DataLoader
full_loader = DataLoader(full_dataset, batch_size=best_hyperparams['batch_size'], shuffle=True, drop_last=True)

# Map selected_features to their indices
selected_feature_indices = [feature_to_idx[feat] for feat in selected_features]

# Select the corresponding adjacency submatrix using integer indices
selected_adjacency = adjacency[np.ix_(selected_feature_indices, selected_feature_indices)]
selected_adjacency_mask = torch.tensor(selected_adjacency, dtype=torch.float32)

device = torch.device('cpu')  # Change to 'cuda' if GPU is available

# Function to initialize and train the model
def train_and_save_model(run_number, seed, model_save_path):
    set_seed(seed)
    
    # Initialize the model
    model = CustomNetwork(num_features=len(selected_features), 
                          adjacency_mask=selected_adjacency_mask).to(device)
    
    # Define loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_hyperparams['lr'], 
                                  weight_decay=best_hyperparams['weight_decay'])
    
    # Training Loop
    model.train()
    for epoch in range(1, best_hyperparams['n_epochs'] + 1):
        epoch_loss = 0.0
        for batch_X, batch_y in full_loader:
            batch_X = batch_X.to(device).float()
            batch_y = batch_y.to(device).float()

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * batch_X.size(0)
        
        epoch_loss /= len(full_loader.dataset)
        
        if epoch % 500 == 0 or epoch == 1:
            print(f"Run {run_number}, Epoch {epoch}/{best_hyperparams['n_epochs']}, Loss: {epoch_loss:.4f}")
    
    print(f"Run {run_number} Training completed.")
    
    # Save the model's state_dict
    torch.save(model.state_dict(), model_save_path)
    print(f"Run {run_number} model saved to {model_save_path}.")

# Train the model 10 times with different seeds and save each model
num_runs = 20
for run in range(1, num_runs + 1):
    print(f"\n=== Starting Run {run}/{num_runs} ===")
    seed = 42 + run  # Different seed for each run
    model_filename = f"custom_network_trained_run_class123_{run}.pth"
    train_and_save_model(run_number=run, seed=seed, model_save_path=model_filename)

Selected Features: ['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'rs10759753', 'rs2761884', 'rs4362400', 'rs3219008', 'rs3218791', 'rs17576', 'rs2289360', 'rs1134170', 'rs2525504', 'rs4919510', 'rs1937810', 'rs1718119', 'rs10484958', 'rs2285053', 'class12_SNP_risk_score', 'rs1800972', 'rs4328262', 'rs1800469', 'rs2306033', 'rs9340799', 'rs17583842', 'rs4244032', 'rs1643821', 'rs1800470', 'rs3045', 'rs4903399', 'rs3753841', 'rs2277698', 'rs18000

In [2]:
# Function to compute route contributions
def compute_route_contributions(model, sample_X, adjacency_mask, selected_features, leaky_relu_negative_slope=0.01):
    """
    Perform manual reconstruction and compute route contributions for a single sample.
    
    Returns:
        contributions: dict mapping route strings to their contributions
        output_sum: float, the manually reconstructed output
    """
    # Extract model parameters
    W = model.W.detach().cpu().numpy()  # Shape: (num_features, num_features)
    bias = model.bias.detach().cpu().numpy()  # Shape: (num_features,)
    batch_norm = model.batch_norm
    bn_weight = batch_norm.weight.detach().cpu().numpy()  # Gamma
    bn_bias = batch_norm.bias.detach().cpu().numpy()      # Beta
    bn_running_mean = batch_norm.running_mean.detach().cpu().numpy()
    bn_running_var = batch_norm.running_var.detach().cpu().numpy()
    bn_eps = batch_norm.eps
    output_weights = model.output_weights.detach().cpu().numpy()  # Shape: (num_features,)
    
    # Convert sample_X to numpy array
    x = sample_X  # Shape: (num_features,)
    
    # Apply the adjacency mask to the weights
    W_masked = W * adjacency_mask.numpy()  # Shape: (num_features, num_features)
    
    # Compute incoming messages using masked weights
    incoming = W_masked.T @ x  # Shape: (num_features,)
    
    node_input = incoming * x  # Shape: (num_features,)
    
    # Apply batch normalization
    node_input_bn = (node_input - bn_running_mean) / np.sqrt(bn_running_var + bn_eps) * bn_weight + bn_bias
    
    # Add bias
    node_input_bn += bias  # Shape: (num_features,)
    
    # Apply Leaky ReLU activation
    node_output = np.where(node_input_bn > 0, node_input_bn, leaky_relu_negative_slope * node_input_bn)  # Shape: (num_features,)
    
    # Multiply by output weights and sum to get the final output
    output_sum = np.sum(node_output * output_weights)  # Should match the model's output
    
    # Initialize a dictionary to store contributions
    contributions = {}
    
    # Iterate over each node j (receiver)
    for j in range(len(selected_features)):
        # Find all i that are connected to j (i sends to j)
        connected_is = np.where(selected_adjacency[:, j] == 1)[0]
        
        if len(connected_is) == 0:
            continue  # No incoming connections to node j
        
        # Compute c_i_j for all connected i
        c_i_js = W_masked[connected_is, j] * x[connected_is] * x[j]  # Shape: (num_connected_is,)
        
        # Sum of c_i_j for node j
        sum_c_j = np.sum(c_i_js)
        
        if sum_c_j == 0:
            # All contributions are from bias
            node_input_j = 0
            node_input_bn_j = (node_input_j - bn_running_mean[j]) / np.sqrt(bn_running_var[j] + bn_eps) * bn_weight[j] + bn_bias[j]
            node_input_bn_j += bias[j]
            activation_j = node_input_bn_j if node_input_bn_j > 0 else leaky_relu_negative_slope * node_input_bn_j
            node_output_j = activation_j * output_weights[j]
            # Assign to Bias route
            route = f"Bias -> {selected_features[j]} -> Output"
            contributions[route] = contributions.get(route, 0) + node_output_j
            continue  # Move to the next node j
    
        # Calculate proportions based on c_i_j
        proportions = c_i_js / sum_c_j  # Shape: (num_connected_is,)
        
        # Compute node_input_j
        node_input_j = sum_c_j
        
        # Apply batch normalization
        node_input_bn_j = (node_input_j - bn_running_mean[j]) / np.sqrt(bn_running_var[j] + bn_eps) * bn_weight[j] + bn_bias[j]
        
        # Add bias
        node_input_bn_j += bias[j]
        
        # Apply Leaky ReLU activation
        activation_j = node_input_bn_j if node_input_bn_j > 0 else leaky_relu_negative_slope * node_input_bn_j
        
        # Multiply by output weights
        node_output_j = activation_j * output_weights[j]
        
        # Distribute the node_output_j among all connected i's based on proportions
        for idx, i in enumerate(connected_is):
            contrib = proportions[idx] * node_output_j
            feature_i = selected_features[i]
            feature_j = selected_features[j]
            route = f"{feature_i} -> {feature_j} -> Output"
            contributions[route] = contributions.get(route, 0) + contrib
    
    # Convert contributions to a DataFrame for better visualization
    contributions_df = pd.DataFrame([
        {'Route': route, 'Contribution': contrib}
        for route, contrib in contributions.items()
    ])
    
    # Sort contributions by absolute value in descending order
    contributions_df['Abs_Contribution'] = contributions_df['Contribution'].abs()
    contributions_df = contributions_df.sort_values(by='Abs_Contribution', ascending=False).drop(columns=['Abs_Contribution'])
    
    return contributions_df, output_sum

# Initialize a dictionary to store cumulative contributions
# Key: route string, Value: dict with 'sum', 'positive', 'negative'
cumulative_contributions = {}

# Function to load a model
def load_model(model_path, num_features, adjacency_mask):
    model = CustomNetwork(num_features=num_features, adjacency_mask=adjacency_mask).to('cpu')
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.eval()
    return model

# Function to evaluate a single sample across all models
def evaluate_sample(sample_idx):
    """
    Evaluate a specific sample across all trained models.

    Args:
        sample_idx (int): Index of the sample to evaluate.

    Returns:
        None. Prints the top 10 positive and top 10 negative routes.
    """
    global cumulative_contributions  # To accumulate contributions across runs

    # Select a specific sample
    sample_X = X_selected.iloc[sample_idx].values.astype(np.float32)

    # Iterate over each run
    num_runs = 20
    for run in range(1, num_runs + 1):
        print(f"\n=== Evaluating Run {run}/{num_runs} ===")
        model_filename = f"custom_network_trained_run_class123_{run}.pth"
        if not os.path.exists(model_filename):
            print(f"Model file {model_filename} not found. Skipping Run {run}.")
            continue

        # Load the model
        model = load_model(model_path=model_filename, num_features=len(selected_features), adjacency_mask=selected_adjacency_mask)

        # Compute manual route contributions
        contributions_df, output_sum = compute_route_contributions(model, sample_X, selected_adjacency_mask, selected_features)

        # Aggregate contributions
        for _, row in contributions_df.iterrows():
            route = row['Route']
            contrib = row['Contribution']
            if route not in cumulative_contributions:
                cumulative_contributions[route] = {'sum': 0.0, 'positive': 0, 'negative': 0}
            cumulative_contributions[route]['sum'] += contrib
            if contrib > 0:
                cumulative_contributions[route]['positive'] += 1
            elif contrib < 0:
                cumulative_contributions[route]['negative'] += 1
            # Zero contributions are ignored

        # Optionally, print the model's output for this run
        with torch.no_grad():
            # Convert sample_X to a PyTorch tensor and reshape to (1, num_features)
            sample_tensor = torch.tensor(sample_X, dtype=torch.float32).unsqueeze(0)  # Shape: (1, num_features)

            # Pass the sample through the model
            model_output = model(sample_tensor).item()  # Get scalar value

        print(f"Run {run} Model's Original Prediction: {model_output}")
        print(f"Run {run} Sum of Route Contributions: {output_sum}")
        print(f"Difference: {abs(output_sum - model_output)}")

    # After all runs, process the cumulative contributions
    contrib_records = []
    for route, data_dict in cumulative_contributions.items():
        contrib_records.append({
            'Route': route,
            'Total_Contribution': data_dict['sum'],
            'Positive_Count': data_dict['positive'],
            'Negative_Count': data_dict['negative']
        })

    contrib_df = pd.DataFrame(contrib_records)

    # Sort to get top 10 positive and top 10 negative routes
    top_positive = contrib_df.sort_values(by='Total_Contribution', ascending=False).head(10)
    top_negative = contrib_df.sort_values(by='Total_Contribution', ascending=True).head(10)

    print("\n=== Top 10 Positive Routes ===")
    print(top_positive[['Route', 'Total_Contribution', 'Positive_Count', 'Negative_Count']])

    print("\n=== Top 10 Negative Routes ===")
    print(top_negative[['Route', 'Total_Contribution', 'Positive_Count', 'Negative_Count']])

    # Additionally, print the number of positives and negatives within the 10 runs for these routes
    print("\n=== Top 10 Positive Routes with Counts ===")
    for idx, row in top_positive.iterrows():
        print(f"Route: {row['Route']}, Total Contribution: {row['Total_Contribution']:.4f}, "
              f"Positive Runs: {int(row['Positive_Count'])}, Negative Runs: {int(row['Negative_Count'])}")

    print("\n=== Top 10 Negative Routes with Counts ===")
    for idx, row in top_negative.iterrows():
        print(f"Route: {row['Route']}, Total Contribution: {row['Total_Contribution']:.4f}, "
              f"Positive Runs: {int(row['Positive_Count'])}, Negative Runs: {int(row['Negative_Count'])}")

# To evaluate the first sample (index 0), uncomment the following line:
evaluate_sample(sample_idx=3607)


=== Evaluating Run 1/20 ===
Run 1 Model's Original Prediction: 0.6762509346008301
Run 1 Sum of Route Contributions: 0.6762520670890808
Difference: 1.1324882507324219e-06

=== Evaluating Run 2/20 ===
Run 2 Model's Original Prediction: 0.521295964717865
Run 2 Sum of Route Contributions: 0.5212950110435486
Difference: 9.5367431640625e-07

=== Evaluating Run 3/20 ===
Run 3 Model's Original Prediction: 0.6014708280563354
Run 3 Sum of Route Contributions: 0.6014704704284668
Difference: 3.5762786865234375e-07

=== Evaluating Run 4/20 ===
Run 4 Model's Original Prediction: 0.7240538597106934
Run 4 Sum of Route Contributions: 0.7240539789199829
Difference: 1.1920928955078125e-07

=== Evaluating Run 5/20 ===
Run 5 Model's Original Prediction: 0.4731309413909912
Run 5 Sum of Route Contributions: 0.4731307029724121
Difference: 2.384185791015625e-07

=== Evaluating Run 6/20 ===
Run 6 Model's Original Prediction: 0.4368837773799896
Run 6 Sum of Route Contributions: 0.43688344955444336
Difference: 3

In [3]:
def compute_node_contributions(model, sample_X, adjacency_mask, selected_features, leaky_relu_negative_slope=0.01):
    """
    Perform manual reconstruction and compute node contributions for a single sample.
    
    Returns:
        node_contributions: dict mapping node names to their contributions
        output_sum: float, the manually reconstructed output
    """
    # Extract model parameters
    W = model.W.detach().cpu().numpy()  # Shape: (num_features, num_features)
    bias = model.bias.detach().cpu().numpy()  # Shape: (num_features,)
    batch_norm = model.batch_norm
    bn_weight = batch_norm.weight.detach().cpu().numpy()  # Gamma
    bn_bias = batch_norm.bias.detach().cpu().numpy()      # Beta
    bn_running_mean = batch_norm.running_mean.detach().cpu().numpy()
    bn_running_var = batch_norm.running_var.detach().cpu().numpy()
    bn_eps = batch_norm.eps
    output_weights = model.output_weights.detach().cpu().numpy()  # Shape: (num_features,)
    
    # Convert sample_X to numpy array
    x = sample_X  # Shape: (num_features,)
    
    # Apply the adjacency mask to the weights
    W_masked = W * adjacency_mask.numpy()  # Shape: (num_features, num_features)
    
    # Compute incoming messages using masked weights
    incoming = W_masked.T @ x  # Shape: (num_features,)
    
    node_input = incoming * x  # Shape: (num_features,)
    
    # Apply batch normalization
    node_input_bn = (node_input - bn_running_mean) / np.sqrt(bn_running_var + bn_eps) * bn_weight + bn_bias
    
    # Add bias
    node_input_bn += bias  # Shape: (num_features,)
    
    # Apply Leaky ReLU activation
    node_output = np.where(node_input_bn > 0, node_input_bn, leaky_relu_negative_slope * node_input_bn)  # Shape: (num_features,)
    
    # Multiply by output weights and sum to get the final output
    output_sum = np.sum(node_output * output_weights)  # Should match the model's output
    
    # Initialize a dictionary to store node contributions
    node_contributions = {}
    
    for j in range(len(selected_features)):
        # Node j's contribution to the output
        contrib = node_output[j] * output_weights[j]
        node_contributions[selected_features[j]] = contrib
    
    return node_contributions, output_sum

def evaluate_node_contributions(sample_idx):
    """
    Evaluate node contributions for a specific sample across all trained models.
    
    Args:
        sample_idx (int): Index of the sample to evaluate.
    
    Returns:
        None. Prints the aggregated node contributions.
    """
    # Initialize a dictionary to store cumulative node contributions
    # Key: node name, Value: dict with 'sum', 'positive', 'negative'
    cumulative_node_contributions = {}
    
    # Select a specific sample
    sample_X = X_selected.iloc[sample_idx].values.astype(np.float32)
    
    num_runs = 20
    for run in range(1, num_runs + 1):
        print(f"\n=== Evaluating Run {run}/{num_runs} ===")
        model_filename = f"custom_network_trained_run_class123_{run}.pth"
        if not os.path.exists(model_filename):
            print(f"Model file {model_filename} not found. Skipping Run {run}.")
            continue

        # Load the model
        model = load_model(model_path=model_filename, num_features=len(selected_features), adjacency_mask=selected_adjacency_mask)

        # Compute node contributions
        node_contrib_dict, output_sum = compute_node_contributions(model, sample_X, selected_adjacency_mask, selected_features)
        
        # Aggregate contributions
        for node, contrib in node_contrib_dict.items():
            if node not in cumulative_node_contributions:
                cumulative_node_contributions[node] = {'sum': 0.0, 'positive': 0, 'negative': 0}
            cumulative_node_contributions[node]['sum'] += contrib
            if contrib > 0:
                cumulative_node_contributions[node]['positive'] += 1
            elif contrib < 0:
                cumulative_node_contributions[node]['negative'] += 1
            # Zero contributions are ignored

        # Optionally, print the model's output for this run
        with torch.no_grad():
            # Convert sample_X to a PyTorch tensor and reshape to (1, num_features)
            sample_tensor = torch.tensor(sample_X, dtype=torch.float32).unsqueeze(0).to(device)  # Shape: (1, num_features)

            # Pass the sample through the model
            model_output = model(sample_tensor).item()  # Get scalar value

        print(f"Run {run} Model's Original Prediction: {model_output}")
        print(f"Run {run} Sum of Node Contributions: {output_sum}")
        print(f"Difference: {abs(output_sum - model_output)}")
    
    # After all runs, process the cumulative node contributions
    contrib_records = []
    for node, data_dict in cumulative_node_contributions.items():
        contrib_records.append({
            'Node': node,
            'Total_Contribution': data_dict['sum'],
            'Positive_Count': data_dict['positive'],
            'Negative_Count': data_dict['negative']
        })
    
    contrib_df = pd.DataFrame(contrib_records)
    
    # Sort to get nodes by Total_Contribution in descending order
    sorted_contrib_df = contrib_df.sort_values(by='Total_Contribution', ascending=False)
    
    # Additionally, print the nodes sorted by Total_Contribution
    print("\n=== Nodes Sorted by Total Contribution (Descending) ===")
    print(sorted_contrib_df[['Node', 'Total_Contribution', 'Positive_Count', 'Negative_Count']].to_string(index=False))

# To evaluate the node contributions for the first sample (index 3769), uncomment the following line:
evaluate_node_contributions(sample_idx=3607)


=== Evaluating Run 1/20 ===
Run 1 Model's Original Prediction: 0.6762509346008301
Run 1 Sum of Node Contributions: 0.6762520670890808
Difference: 1.1324882507324219e-06

=== Evaluating Run 2/20 ===
Run 2 Model's Original Prediction: 0.521295964717865
Run 2 Sum of Node Contributions: 0.5212950110435486
Difference: 9.5367431640625e-07

=== Evaluating Run 3/20 ===
Run 3 Model's Original Prediction: 0.6014708280563354
Run 3 Sum of Node Contributions: 0.6014704704284668
Difference: 3.5762786865234375e-07

=== Evaluating Run 4/20 ===
Run 4 Model's Original Prediction: 0.7240538597106934
Run 4 Sum of Node Contributions: 0.7240539789199829
Difference: 1.1920928955078125e-07

=== Evaluating Run 5/20 ===
Run 5 Model's Original Prediction: 0.4731309413909912
Run 5 Sum of Node Contributions: 0.4731307029724121
Difference: 2.384185791015625e-07

=== Evaluating Run 6/20 ===
Run 6 Model's Original Prediction: 0.4368837773799896
Run 6 Sum of Node Contributions: 0.43688344955444336
Difference: 3.27825

In [4]:
# Function to analyze consistent signs of W[i,j] and output_weights[j] across runs
def analyze_consistent_route_signs(selected_features, selected_adjacency_mask, num_runs=20):
    """
    Analyze and identify routes with consistent signs across multiple model runs.
    For each route (i -> j -> Output), both W[i,j] and output_weights[j] must have the same sign across all runs.

    Args:
        selected_features (list): List of selected feature names.
        selected_adjacency_mask (torch.Tensor): Adjacency mask tensor for selected features.
        num_runs (int): Number of trained model runs to analyze.

    Returns:
        None. Prints the consistent routes and their consistent signs.
    """
    # Initialize dictionaries to store signs
    # Key: route (i -> j -> Output)
    # Values: list of signs across runs
    W_signs = {}
    output_weight_signs = {}

    # Get the adjacency as a NumPy array
    adjacency_np = selected_adjacency_mask.numpy()

    # Iterate over each run
    for run in range(1, num_runs + 1):
        model_filename = f"custom_network_trained_run_class123_{run}.pth"
        if not os.path.exists(model_filename):
            print(f"Model file {model_filename} not found. Skipping Run {run}.")
            continue

        # Load the model
        model = CustomNetwork(num_features=len(selected_features), adjacency_mask=selected_adjacency_mask)
        model.load_state_dict(torch.load(model_filename, map_location='cpu'))
        model.eval()

        # Extract W and output_weights
        with torch.no_grad():
            W = model.W.cpu().numpy()  # Shape: (num_features, num_features)
            output_weights = model.output_weights.cpu().numpy()  # Shape: (num_features,)

        # Iterate over all possible routes based on adjacency
        for j in range(len(selected_features)):  # Receiver node
            for i in range(len(selected_features)):  # Sender node
                if adjacency_np[j, i] == 1.0:
                    route = f"{selected_features[i]} -> {selected_features[j]} -> Output"

                    # Initialize lists if not already
                    if route not in W_signs:
                        W_signs[route] = []
                    if route not in output_weight_signs:
                        output_weight_signs[route] = []

                    # Record the sign of W[i,j]
                    W_ij = W[i, j]
                    W_sign = np.sign(W_ij)
                    W_signs[route].append(W_sign)

                    # Record the sign of output_weights[j]
                    output_w_j = output_weights[j]
                    output_sign = np.sign(output_w_j)
                    output_weight_signs[route].append(output_sign)

    # Identify routes with consistent W[i,j] and output_weights[j] signs across all runs
    consistent_routes = []

    for route in W_signs:
        # Check if both W[i,j] and output_weights[j] have consistent signs across runs
        W_sign_list = W_signs[route]
        output_sign_list = output_weight_signs[route]

        if len(W_sign_list) != num_runs or len(output_sign_list) != num_runs:
            # Incomplete data for this route across all runs
            continue

        # Check if all W[i,j] signs are the same and non-zero
        first_W_sign = W_sign_list[0]
        W_consistent = all(sign == first_W_sign for sign in W_sign_list) and first_W_sign != 0

        # Check if all output_weights[j] signs are the same and non-zero
        first_output_sign = output_sign_list[0]
        output_consistent = all(sign == first_output_sign for sign in output_sign_list) and first_output_sign != 0

        if W_consistent and output_consistent:
            consistent_routes.append({
                'Route': route,
                'W[i,j] Sign': 'Positive' if first_W_sign > 0 else 'Negative',
                'output_weights[j] Sign': 'Positive' if first_output_sign > 0 else 'Negative'
            })

    # Convert to DataFrame for better visualization
    consistent_routes_df = pd.DataFrame(consistent_routes)

    # Sort routes by Route name for readability
    consistent_routes_df = consistent_routes_df.sort_values(by=['Route'])

    # Print the consistent routes
    print("\n=== Routes with Consistent Signs Across All Runs ===")
    if consistent_routes_df.empty:
        print("No routes have consistent W[i,j] and output_weights[j] signs across all runs.")
    else:
        print(consistent_routes_df.to_string(index=False))

# Example usage:
# Place this block after your existing training loop
analyze_consistent_route_signs(
    selected_features=selected_features,
    selected_adjacency_mask=selected_adjacency_mask,
    num_runs=20
)


=== Routes with Consistent Signs Across All Runs ===
                                                                               Route W[i,j] Sign output_weights[j] Sign
                                                BMD_body -> Flight_time_10 -> Output    Positive               Negative
                                      BMD_spine -> knee_flexion_peak_angle -> Output    Positive               Negative
                                                    BMD_spine -> rs4454832 -> Output    Positive               Negative
                                                  BMI -> Step_frequency_12 -> Output    Negative               Negative
                                       Duty_factor_10 -> Step_frequency_12 -> Output    Positive               Negative
                                       Duty_factor_asymmetry_10 -> BMD_hip -> Output    Negative               Negative
                                Duty_factor_asymmetry_10 -> Impact_peak_12 -> Output    Negative          

In [6]:
import shap
from tqdm import tqdm
from scipy.stats import spearmanr

def calculate_jaccard(set1, set2):
    """Calculate Jaccard similarity between two sets."""
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union != 0 else 0

def calculate_rank_correlation(rank1, rank2):
    """Calculate Spearman correlation between two ranked lists."""
    common_items = list(set(rank1) & set(rank2))
    if len(common_items) < 2:
        return np.nan
    rank1_pos = [rank1.index(item) for item in common_items]
    rank2_pos = [rank2.index(item) for item in common_items]
    return spearmanr(rank1_pos, rank2_pos).correlation

def calculate_correlations(subset_df):
    """Calculate Pearson and Spearman correlations with error handling."""
    try:
        contrib_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='pearson').iloc[0,1]
    except:
        contrib_corr = np.nan
    try:
        ratio_corr = subset_df[['Node_Positive_Ratio', 'SHAP_Positive_Ratio']].corr(method='pearson').iloc[0,1]
    except:
        ratio_corr = np.nan
    try:
        rank_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='spearman').iloc[0,1]
    except:
        rank_corr = np.nan
    return contrib_corr, ratio_corr, rank_corr

def compute_node_contributions(model, sample_X, adjacency_mask, selected_features, leaky_relu_negative_slope=0.01):
    """
    Perform manual reconstruction and compute node contributions for a single sample.
    
    Returns:
        node_contributions: dict mapping node names to their contributions
        output_sum: float, the manually reconstructed output
    """
    # Extract model parameters
    W = model.W.detach().cpu().numpy()  # Shape: (num_features, num_features)
    bias = model.bias.detach().cpu().numpy()  # Shape: (num_features,)
    batch_norm = model.batch_norm
    bn_weight = batch_norm.weight.detach().cpu().numpy()  # Gamma
    bn_bias = batch_norm.bias.detach().cpu().numpy()      # Beta
    bn_running_mean = batch_norm.running_mean.detach().cpu().numpy()
    bn_running_var = batch_norm.running_var.detach().cpu().numpy()
    bn_eps = batch_norm.eps
    output_weights = model.output_weights.detach().cpu().numpy()  # Shape: (num_features,)
    
    # Convert sample_X to numpy array
    x = sample_X  # Shape: (num_features,)
    
    # Apply the adjacency mask to the weights
    W_masked = W * adjacency_mask.numpy()  # Shape: (num_features, num_features)
    
    # Compute incoming messages using masked weights
    incoming = W_masked.T @ x  # Shape: (num_features,)
    
    node_input = incoming * x  # Shape: (num_features,)
    
    # Apply batch normalization
    node_input_bn = (node_input - bn_running_mean) / np.sqrt(bn_running_var + bn_eps) * bn_weight + bn_bias
    
    # Add bias
    node_input_bn += bias  # Shape: (num_features,)
    
    # Apply Leaky ReLU activation
    node_output = np.where(node_input_bn > 0, node_input_bn, leaky_relu_negative_slope * node_input_bn)  # Shape: (num_features,)
    
    # Multiply by output weights and sum to get the final output
    output_sum = np.sum(node_output * output_weights)  # Should match the model's output
    
    # Initialize a dictionary to store node contributions
    node_contributions = {}
    
    for j in range(len(selected_features)):
        # Node j's contribution to the output
        contrib = node_output[j] * output_weights[j]
        node_contributions[selected_features[j]] = contrib
    
    return node_contributions, output_sum

# Configuration
num_runs = 20
num_samples = 100
np.random.seed(42)
sample_indices = np.random.choice(len(X_selected), size=num_samples, replace=False)
background = shap.sample(X_selected, 100, random_state=42).values.astype(np.float32)

# Initialize results storage
all_run_results = []

for run in tqdm(range(1, num_runs + 1), desc="Processing runs"):
    # Load model for this run
    model_path = f"custom_network_trained_run_class123_{run}.pth"
    model = load_model(model_path, len(selected_features), selected_adjacency_mask)
    model.eval()
    
    # Initialize storage for this run's correlations
    run_correlations = {
        'all_features_contrib': [],
        'all_features_ratio': [],
        'all_features_rank': [],
        'top_node_5pos5neg_contrib': [],
        'top_node_5pos5neg_ratio': [],
        'top_node_5pos5neg_rank': [],
        'top_shap_5pos5neg_contrib': [],
        'top_shap_5pos5neg_ratio': [],
        'top_shap_5pos5neg_rank': [],
        'combined_3n3s_contrib': [],
        'combined_3n3s_ratio': [],
        'combined_3n3s_rank': [],
        'top_node_ratio5_contrib': [],
        'top_node_ratio5_ratio': [],
        'top_node_ratio5_rank': [],
        'top_shap_ratio5_contrib': [],
        'top_shap_ratio5_ratio': [],
        'top_shap_ratio5_rank': [],
        'combined_ratio3_contrib': [],
        'combined_ratio3_ratio': [],
        'combined_ratio3_rank': []
    }

    # Process each sample in this run
    for sample_idx in sample_indices:
        sample_X = X_selected.iloc[sample_idx].values.astype(np.float32)
        sample_to_explain = sample_X.reshape(1, -1)

        # Compute SHAP values for this run-sample combination
        def model_predict(data):
            tensor_data = torch.tensor(data, dtype=torch.float32)
            with torch.no_grad():
                outputs = model(tensor_data)
            return outputs.numpy()

        explainer = shap.KernelExplainer(model_predict, background)
        shap_values = explainer.shap_values(sample_to_explain, silent=True)[0]

        # Compute node contributions for this run-sample combination
        node_contrib, _ = compute_node_contributions(model, sample_X, 
                                                   selected_adjacency_mask, 
                                                   selected_features)

        # Create combined DataFrame for this run-sample
        combined_df = pd.DataFrame({
            'Feature': selected_features,
            'Average_SHAP': shap_values,
            'Node_Contribution': [node_contrib[f] for f in selected_features],
            'SHAP_Positive': (shap_values > 0).astype(int),
            'SHAP_Negative': (shap_values < 0).astype(int),
            # Fix: Convert list to numpy array before using astype()
            'Node_Positive': np.array([v > 0 for v in node_contrib.values()]).astype(int),
            'Node_Negative': np.array([v < 0 for v in node_contrib.values()]).astype(int)
        })

        # Calculate positive ratios
        combined_df['Node_Positive_Ratio'] = combined_df.apply(
            lambda x: x['Node_Positive'] / (x['Node_Positive'] + x['Node_Negative']) 
            if (x['Node_Positive'] + x['Node_Negative']) > 0 else 0.5, axis=1)
        
        combined_df['SHAP_Positive_Ratio'] = combined_df.apply(
            lambda x: x['SHAP_Positive'] / (x['SHAP_Positive'] + x['SHAP_Negative']) 
            if (x['SHAP_Positive'] + x['SHAP_Negative']) > 0 else 0.5, axis=1)

        # Get feature rankings for this run-sample
        node_rank = combined_df.sort_values('Node_Contribution', 
                                          ascending=False)['Feature'].tolist()
        shap_rank = combined_df.sort_values('Average_SHAP', 
                                          ascending=False)['Feature'].tolist()

        # Calculate correlations for different feature sets
        # 1. All features
        c, r, s = calculate_correlations(combined_df)
        run_correlations['all_features_contrib'].append(c)
        run_correlations['all_features_ratio'].append(r)
        run_correlations['all_features_rank'].append(s)

        # 2. Top 5 node pos + 5 node neg
        top_node = node_rank[:5] + node_rank[-5:]
        subset = combined_df[combined_df['Feature'].isin(top_node)]
        c, r, s = calculate_correlations(subset)
        run_correlations['top_node_5pos5neg_contrib'].append(c)
        run_correlations['top_node_5pos5neg_ratio'].append(r)
        run_correlations['top_node_5pos5neg_rank'].append(s)

        # 3. Top 5 SHAP pos + 5 SHAP neg
        top_shap = shap_rank[:5] + shap_rank[-5:]
        subset = combined_df[combined_df['Feature'].isin(top_shap)]
        c, r, s = calculate_correlations(subset)
        run_correlations['top_shap_5pos5neg_contrib'].append(c)
        run_correlations['top_shap_5pos5neg_ratio'].append(r)
        run_correlations['top_shap_5pos5neg_rank'].append(s)

        # 4. Combined 3n+3s
        combined_features = top_node[:3] + top_node[-3:] + top_shap[:3] + top_shap[-3:]
        subset = combined_df[combined_df['Feature'].isin(combined_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['combined_3n3s_contrib'].append(c)
        run_correlations['combined_3n3s_ratio'].append(r)
        run_correlations['combined_3n3s_rank'].append(s)

        # 5. Top node ratios (weighted by 5)
        node_ratio_features = combined_df.sort_values('Node_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                            combined_df.sort_values('Node_Positive_Ratio').head(5)['Feature'].tolist()
        subset = combined_df[combined_df['Feature'].isin(node_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['top_node_ratio5_contrib'].append(c)
        run_correlations['top_node_ratio5_ratio'].append(r)
        run_correlations['top_node_ratio5_rank'].append(s)

        # 6. Top SHAP ratios (weighted by 3)
        shap_ratio_features = combined_df.sort_values('SHAP_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                            combined_df.sort_values('SHAP_Positive_Ratio').head(5)['Feature'].tolist()
        subset = combined_df[combined_df['Feature'].isin(shap_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['top_shap_ratio5_contrib'].append(c)
        run_correlations['top_shap_ratio5_ratio'].append(r)
        run_correlations['top_shap_ratio5_rank'].append(s)

        # 7. Combined ratios
        combined_ratio_features = node_ratio_features[:3] + node_ratio_features[-3:] + shap_ratio_features[:3] + shap_ratio_features[-3:]
        subset = combined_df[combined_df['Feature'].isin(combined_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['combined_ratio3_contrib'].append(c)
        run_correlations['combined_ratio3_ratio'].append(r)
        run_correlations['combined_ratio3_rank'].append(s)

    # Average correlations across samples for this run
    averaged_run = {k: np.nanmean(v) for k, v in run_correlations.items()}
    all_run_results.append(averaged_run)

# Create final summary DataFrame
summary_df = pd.DataFrame(all_run_results)
final_summary = pd.DataFrame()

for col in summary_df.columns:
    final_summary[col] = [
        summary_df[col].mean(),
        summary_df[col].std(),
        summary_df[col].median(),
        summary_df[col].quantile(0.25),
        summary_df[col].quantile(0.75)
    ]

final_summary.index = ['mean', 'std', 'median', '25%', '75%']
print(final_summary.transpose())

Processing runs: 100%|██████████████████████████████████████████████████████████████| 20/20 [1:37:14<00:00, 291.73s/it]

                               mean       std    median       25%       75%
all_features_contrib       0.062470  0.015823  0.062518  0.049742  0.069848
all_features_ratio         0.046754  0.025169  0.050073  0.028805  0.061039
all_features_rank          0.032202  0.030311  0.029360  0.013974  0.052285
top_node_5pos5neg_contrib  0.184683  0.067575  0.182721  0.137348  0.231127
top_node_5pos5neg_ratio    0.134894  0.082253  0.141969  0.083883  0.177915
top_node_5pos5neg_rank     0.174121  0.076872  0.163382  0.107037  0.237142
top_shap_5pos5neg_contrib  0.114218  0.053156  0.107653  0.077997  0.139265
top_shap_5pos5neg_ratio    0.175648  0.071734  0.188565  0.099425  0.226539
top_shap_5pos5neg_rank     0.130673  0.075144  0.115152  0.070697  0.182576
combined_3n3s_contrib      0.073305  0.027248  0.068626  0.049786  0.087037
combined_3n3s_ratio        0.170877  0.058560  0.167837  0.123362  0.225229
combined_3n3s_rank         0.119941  0.049552  0.111271  0.081841  0.164142
top_node_rat

In [7]:
import shap
from tqdm import tqdm
from scipy.stats import spearmanr

def calculate_jaccard(set1, set2):
    """Calculate Jaccard similarity between two sets."""
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union != 0 else 0

def calculate_rank_correlation(rank1, rank2):
    """Calculate Spearman correlation between two ranked lists."""
    common_items = list(set(rank1) & set(rank2))
    if len(common_items) < 2:
        return np.nan
    rank1_pos = [rank1.index(item) for item in common_items]
    rank2_pos = [rank2.index(item) for item in common_items]
    return spearmanr(rank1_pos, rank2_pos).correlation

def calculate_correlations(subset_df):
    """Calculate Pearson and Spearman correlations with error handling."""
    try:
        contrib_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='pearson').iloc[0,1]
    except:
        contrib_corr = np.nan
    try:
        ratio_corr = subset_df[['Node_Positive_Ratio', 'SHAP_Positive_Ratio']].corr(method='pearson').iloc[0,1]
    except:
        ratio_corr = np.nan
    try:
        rank_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='spearman').iloc[0,1]
    except:
        rank_corr = np.nan
    return contrib_corr, ratio_corr, rank_corr

def compute_node_contributions(model, sample_X, adjacency_mask, selected_features, leaky_relu_negative_slope=0.01):
    # Extract model parameters
    W = model.W.detach().cpu().numpy()
    bias = model.bias.detach().cpu().numpy()
    batch_norm = model.batch_norm
    bn_weight = batch_norm.weight.detach().cpu().numpy()
    bn_bias = batch_norm.bias.detach().cpu().numpy()
    bn_running_mean = batch_norm.running_mean.detach().cpu().numpy()
    bn_running_var = batch_norm.running_var.detach().cpu().numpy()
    bn_eps = batch_norm.eps
    output_weights = model.output_weights.detach().cpu().numpy()
    
    x = sample_X
    num_features = len(selected_features)
    adjacency = adjacency_mask.numpy()

    # Precompute node inputs and activations
    node_inputs = np.zeros(num_features)
    for k in range(num_features):
        connected_i = np.where(adjacency[:, k] == 1)[0]
        node_inputs[k] = np.sum(W[connected_i, k] * x[connected_i] * x[k])

    # Process through batch norm and activation
    node_inputs_bn = (node_inputs - bn_running_mean) / np.sqrt(bn_running_var + bn_eps) * bn_weight + bn_bias
    node_inputs_bn += bias
    activations = np.where(node_inputs_bn > 0, node_inputs_bn, leaky_relu_negative_slope * node_inputs_bn)
    
    # Direct contributions (i->j->output)
    direct_contributions = activations * output_weights

    # Indirect contributions (j->k->output)
    indirect_contributions = np.zeros(num_features)
    for j in range(num_features):
        # Find all k that j connects to (outgoing)
        connected_ks = np.where(adjacency[j, :] == 1)[0]
        for k in connected_ks:
            # Calculate j's contribution to k's node input
            j_contribution = W[j, k] * x[j] * x[k]
            total_k_input = node_inputs[k]
            
            if total_k_input == 0:
                proportion = 0
            else:
                proportion = j_contribution / total_k_input
                
            # Attribute proportion of k's contribution back to j
            indirect_contributions[j] += proportion * direct_contributions[k]

    # Total contributions = direct + indirect
    total_contributions = direct_contributions + indirect_contributions

    # Convert to dictionary
    node_contrib_dict = {
        feature: total_contributions[i]
        for i, feature in enumerate(selected_features)
    }
    
    return node_contrib_dict, np.sum(total_contributions)

# Configuration
num_runs = 20
num_samples = 100
np.random.seed(42)
sample_indices = np.random.choice(len(X_selected), size=num_samples, replace=False)
background = shap.sample(X_selected, 100, random_state=42).values.astype(np.float32)

# Initialize results storage
all_run_results = []

for run in tqdm(range(1, num_runs + 1), desc="Processing runs"):
    # Load model for this run
    model_path = f"custom_network_trained_run_class123_{run}.pth"
    model = load_model(model_path, len(selected_features), selected_adjacency_mask)
    model.eval()
    
    # Initialize storage for this run's correlations
    run_correlations = {
        'all_features_contrib': [],
        'all_features_ratio': [],
        'all_features_rank': [],
        'top_node_5pos5neg_contrib': [],
        'top_node_5pos5neg_ratio': [],
        'top_node_5pos5neg_rank': [],
        'top_shap_5pos5neg_contrib': [],
        'top_shap_5pos5neg_ratio': [],
        'top_shap_5pos5neg_rank': [],
        'combined_3n3s_contrib': [],
        'combined_3n3s_ratio': [],
        'combined_3n3s_rank': [],
        'top_node_ratio5_contrib': [],
        'top_node_ratio5_ratio': [],
        'top_node_ratio5_rank': [],
        'top_shap_ratio5_contrib': [],
        'top_shap_ratio5_ratio': [],
        'top_shap_ratio5_rank': [],
        'combined_ratio3_contrib': [],
        'combined_ratio3_ratio': [],
        'combined_ratio3_rank': []
    }

    # Process each sample in this run
    for sample_idx in sample_indices:
        sample_X = X_selected.iloc[sample_idx].values.astype(np.float32)
        sample_to_explain = sample_X.reshape(1, -1)

        # Compute SHAP values for this run-sample combination
        def model_predict(data):
            tensor_data = torch.tensor(data, dtype=torch.float32)
            with torch.no_grad():
                outputs = model(tensor_data)
            return outputs.numpy()

        explainer = shap.KernelExplainer(model_predict, background)
        shap_values = explainer.shap_values(sample_to_explain, silent=True)[0]

        # Compute node contributions for this run-sample combination
        node_contrib, _ = compute_node_contributions(model, sample_X, 
                                                   selected_adjacency_mask, 
                                                   selected_features)

        # Create combined DataFrame for this run-sample
        combined_df = pd.DataFrame({
            'Feature': selected_features,
            'Average_SHAP': shap_values,
            'Node_Contribution': [node_contrib[f] for f in selected_features],
            'SHAP_Positive': (shap_values > 0).astype(int),
            'SHAP_Negative': (shap_values < 0).astype(int),
            # Fix: Convert list to numpy array before using astype()
            'Node_Positive': np.array([v > 0 for v in node_contrib.values()]).astype(int),
            'Node_Negative': np.array([v < 0 for v in node_contrib.values()]).astype(int)
        })

        # Calculate positive ratios
        combined_df['Node_Positive_Ratio'] = combined_df.apply(
            lambda x: x['Node_Positive'] / (x['Node_Positive'] + x['Node_Negative']) 
            if (x['Node_Positive'] + x['Node_Negative']) > 0 else 0.5, axis=1)
        
        combined_df['SHAP_Positive_Ratio'] = combined_df.apply(
            lambda x: x['SHAP_Positive'] / (x['SHAP_Positive'] + x['SHAP_Negative']) 
            if (x['SHAP_Positive'] + x['SHAP_Negative']) > 0 else 0.5, axis=1)

        # Get feature rankings for this run-sample
        node_rank = combined_df.sort_values('Node_Contribution', 
                                          ascending=False)['Feature'].tolist()
        shap_rank = combined_df.sort_values('Average_SHAP', 
                                          ascending=False)['Feature'].tolist()

        # Calculate correlations for different feature sets
        # 1. All features
        c, r, s = calculate_correlations(combined_df)
        run_correlations['all_features_contrib'].append(c)
        run_correlations['all_features_ratio'].append(r)
        run_correlations['all_features_rank'].append(s)

        # 2. Top 5 node pos + 5 node neg
        top_node = node_rank[:5] + node_rank[-5:]
        subset = combined_df[combined_df['Feature'].isin(top_node)]
        c, r, s = calculate_correlations(subset)
        run_correlations['top_node_5pos5neg_contrib'].append(c)
        run_correlations['top_node_5pos5neg_ratio'].append(r)
        run_correlations['top_node_5pos5neg_rank'].append(s)

        # 3. Top 5 SHAP pos + 5 SHAP neg
        top_shap = shap_rank[:5] + shap_rank[-5:]
        subset = combined_df[combined_df['Feature'].isin(top_shap)]
        c, r, s = calculate_correlations(subset)
        run_correlations['top_shap_5pos5neg_contrib'].append(c)
        run_correlations['top_shap_5pos5neg_ratio'].append(r)
        run_correlations['top_shap_5pos5neg_rank'].append(s)

        # 4. Combined 3n+3s
        combined_features = top_node[:3] + top_node[-3:] + top_shap[:3] + top_shap[-3:]
        subset = combined_df[combined_df['Feature'].isin(combined_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['combined_3n3s_contrib'].append(c)
        run_correlations['combined_3n3s_ratio'].append(r)
        run_correlations['combined_3n3s_rank'].append(s)

        # 5. Top node ratios (weighted by 5)
        node_ratio_features = combined_df.sort_values('Node_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                            combined_df.sort_values('Node_Positive_Ratio').head(5)['Feature'].tolist()
        subset = combined_df[combined_df['Feature'].isin(node_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['top_node_ratio5_contrib'].append(c)
        run_correlations['top_node_ratio5_ratio'].append(r)
        run_correlations['top_node_ratio5_rank'].append(s)

        # 6. Top SHAP ratios (weighted by 3)
        shap_ratio_features = combined_df.sort_values('SHAP_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                            combined_df.sort_values('SHAP_Positive_Ratio').head(5)['Feature'].tolist()
        subset = combined_df[combined_df['Feature'].isin(shap_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['top_shap_ratio5_contrib'].append(c)
        run_correlations['top_shap_ratio5_ratio'].append(r)
        run_correlations['top_shap_ratio5_rank'].append(s)

        # 7. Combined ratios
        combined_ratio_features = node_ratio_features[:3] + node_ratio_features[-3:] + shap_ratio_features[:3] + shap_ratio_features[-3:]
        subset = combined_df[combined_df['Feature'].isin(combined_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['combined_ratio3_contrib'].append(c)
        run_correlations['combined_ratio3_ratio'].append(r)
        run_correlations['combined_ratio3_rank'].append(s)

    # Average correlations across samples for this run
    averaged_run = {k: np.nanmean(v) for k, v in run_correlations.items()}
    all_run_results.append(averaged_run)

# Create final summary DataFrame
summary_df = pd.DataFrame(all_run_results)
final_summary = pd.DataFrame()

for col in summary_df.columns:
    final_summary[col] = [
        summary_df[col].mean(),
        summary_df[col].std(),
        summary_df[col].median(),
        summary_df[col].quantile(0.25),
        summary_df[col].quantile(0.75)
    ]

final_summary.index = ['mean', 'std', 'median', '25%', '75%']
print(final_summary.transpose())

Processing runs: 100%|██████████████████████████████████████████████████████████████| 20/20 [1:24:39<00:00, 253.95s/it]

                               mean       std    median       25%       75%
all_features_contrib       0.256859  0.039768  0.259176  0.228314  0.282198
all_features_ratio         0.134581  0.016251  0.136097  0.123704  0.144167
all_features_rank          0.149564  0.021133  0.151585  0.137820  0.166466
top_node_5pos5neg_contrib  0.471239  0.078570  0.481293  0.418926  0.511024
top_node_5pos5neg_ratio    0.490552  0.078011  0.506089  0.453733  0.530617
top_node_5pos5neg_rank     0.465905  0.077759  0.480512  0.416135  0.509181
top_shap_5pos5neg_contrib  0.451316  0.073886  0.466973  0.410034  0.501112
top_shap_5pos5neg_ratio    0.449688  0.054076  0.451107  0.405729  0.496012
top_shap_5pos5neg_rank     0.424879  0.065178  0.434121  0.372515  0.471242
combined_3n3s_contrib      0.357194  0.055783  0.364234  0.325849  0.394739
combined_3n3s_ratio        0.481141  0.054726  0.483314  0.446339  0.516357
combined_3n3s_rank         0.343220  0.054688  0.359140  0.307606  0.379792
top_node_rat

In [11]:
def compute_route_contributions(model, sample_X, adjacency_mask, selected_features, leaky_relu_negative_slope=0.01):
    # Extract model parameters
    W = model.W.detach().cpu().numpy()
    bias = model.bias.detach().cpu().numpy()
    batch_norm = model.batch_norm
    gamma = batch_norm.weight.detach().cpu().numpy()
    beta = batch_norm.bias.detach().cpu().numpy()
    running_mean = batch_norm.running_mean.detach().cpu().numpy()
    running_var = batch_norm.running_var.detach().cpu().numpy()
    eps = batch_norm.eps
    output_weights = model.output_weights.detach().cpu().numpy()
    
    num_features = len(selected_features)
    adjacency = adjacency_mask.numpy()
    x = sample_X  # Shape: (num_features,)
    
    # Precompute activation slopes and bias contributions for each node j
    activation_slopes = np.zeros(num_features)
    bias_contributions = np.zeros(num_features)
    for j in range(num_features):
        connected_i = np.where(adjacency[:, j] == 1)[0]
        sum_terms_j = np.sum(x[connected_i] * W[connected_i, j] * x[j])
        
        # Compute BN scaled terms and total input
        bn_scaled_j = (sum_terms_j - running_mean[j]) / np.sqrt(running_var[j] + eps) * gamma[j] + beta[j]
        total_input_j = bn_scaled_j + bias[j]
        activation_slope = 1.0 if total_input_j > 0 else leaky_relu_negative_slope
        activation_slopes[j] = activation_slope
        
        # Compute bias contribution for node j
        bias_term = (beta[j] - (running_mean[j] * gamma[j]) / np.sqrt(running_var[j] + eps)) + bias[j]
        bias_contributions[j] = bias_term * activation_slope * output_weights[j]
    
    # Calculate feature route contributions
    contributions = {}
    for i in range(num_features):
        connected_js = np.where(adjacency[i, :] == 1)[0]
        for j in connected_js:
            term = x[i] * W[i, j] * x[j]
            gamma_j = gamma[j]
            sqrt_var_j = np.sqrt(running_var[j] + eps)
            contrib = term * (gamma_j / sqrt_var_j) * activation_slopes[j] * output_weights[j]
            route = f"{selected_features[i]} -> {selected_features[j]} -> Output"
            contributions[route] = contrib
    
    # Add bias contributions
    for j in range(num_features):
        if bias_contributions[j] != 0:
            route = f"Bias -> {selected_features[j]} -> Output"
            contributions[route] = bias_contributions[j]
    
    contributions_df = pd.DataFrame(list(contributions.items()), columns=['Route', 'Contribution'])
    contributions_df = contributions_df.sort_values(by='Contribution', key=abs, ascending=False)
    total_output = contributions_df['Contribution'].sum()
    
    return contributions_df, total_output

import shap
from tqdm import tqdm
from scipy.stats import spearmanr

def calculate_jaccard(set1, set2):
    """Calculate Jaccard similarity between two sets."""
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union != 0 else 0

def calculate_rank_correlation(rank1, rank2):
    """Calculate Spearman correlation between two ranked lists."""
    common_items = list(set(rank1) & set(rank2))
    if len(common_items) < 2:
        return np.nan
    rank1_pos = [rank1.index(item) for item in common_items]
    rank2_pos = [rank2.index(item) for item in common_items]
    return spearmanr(rank1_pos, rank2_pos).correlation

def calculate_correlations(subset_df):
    """Calculate Pearson and Spearman correlations with error handling."""
    try:
        contrib_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='pearson').iloc[0,1]
    except:
        contrib_corr = np.nan
    try:
        ratio_corr = subset_df[['Node_Positive_Ratio', 'SHAP_Positive_Ratio']].corr(method='pearson').iloc[0,1]
    except:
        ratio_corr = np.nan
    try:
        rank_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='spearman').iloc[0,1]
    except:
        rank_corr = np.nan
    return contrib_corr, ratio_corr, rank_corr

def compute_node_contributions(model, sample_X, adjacency_mask, selected_features, leaky_relu_negative_slope=0.01):
    # Extract parameters
    W = model.W.detach().cpu().numpy()
    bias = model.bias.detach().cpu().numpy()
    batch_norm = model.batch_norm
    gamma = batch_norm.weight.detach().cpu().numpy()
    beta = batch_norm.bias.detach().cpu().numpy()
    running_mean = batch_norm.running_mean.detach().cpu().numpy()
    running_var = batch_norm.running_var.detach().cpu().numpy()
    eps = batch_norm.eps
    output_weights = model.output_weights.detach().cpu().numpy()
    
    num_features = len(selected_features)
    adjacency = adjacency_mask.numpy()
    x = sample_X
    
    # Precompute node activations and slopes
    activations = np.zeros(num_features)
    activation_slopes = np.zeros(num_features)
    
    for j in range(num_features):
        # Calculate total input to node j
        connected_i = np.where(adjacency[:, j] == 1)[0]
        sum_terms = np.sum(x[connected_i] * W[connected_i, j] * x[j])
        
        # Batch normalization
        bn_scaled = (sum_terms - running_mean[j]) / np.sqrt(running_var[j] + eps) * gamma[j] + beta[j]
        
        # Add bias
        total_input = bn_scaled + bias[j]
        
        # Store activation and slope
        activations[j] = F.leaky_relu(torch.tensor(total_input), negative_slope=leaky_relu_negative_slope).item()
        activation_slopes[j] = 1.0 if total_input > 0 else leaky_relu_negative_slope

    # Calculate contributions
    node_contributions = np.zeros(num_features)
    
    # 1. Direct contribution: activation -> output
    direct_contrib = activations * output_weights
    
    # 2. Indirect contributions: j -> i -> output
    indirect_contrib = np.zeros(num_features)
    for j in range(num_features):
        # Nodes i that j connects to (j->i)
        connected_i = np.where(adjacency[:, j] == 1)[0]
        
        for i in connected_i:
            # Contribution from j to i's activation
            contrib_j_to_i = x[j] * W[i, j] * x[i] 
            
            # BN scaling for node i
            bn_scale_i = gamma[i] / np.sqrt(running_var[i] + eps)
            
            # Final contribution: j->i * activation slope of i * output weight of i
            indirect_contrib[j] += contrib_j_to_i * bn_scale_i * activation_slopes[i] * output_weights[i]

    # Total contribution = direct + indirect
    total_contrib = direct_contrib + indirect_contrib
    
    # Convert to dictionary
    node_contrib_dict = {selected_features[j]: total_contrib[j] for j in range(num_features)}
    total_output = np.sum(total_contrib)  # Should match model output
    
    return node_contrib_dict, total_output


# Configuration
num_runs = 20
num_samples = 100
np.random.seed(42)
sample_indices = np.random.choice(len(X_selected), size=num_samples, replace=False)
background = shap.sample(X_selected, 100, random_state=42).values.astype(np.float32)

# Initialize results storage
all_run_results = []

for run in tqdm(range(1, num_runs + 1), desc="Processing runs"):
    # Load model for this run
    model_path = f"custom_network_trained_run_class123_{run}.pth"
    model = load_model(model_path, len(selected_features), selected_adjacency_mask)
    model.eval()
    
    # Initialize storage for this run's correlations
    run_correlations = {
        'all_features_contrib': [],
        'all_features_ratio': [],
        'all_features_rank': [],
        'top_node_5pos5neg_contrib': [],
        'top_node_5pos5neg_ratio': [],
        'top_node_5pos5neg_rank': [],
        'top_shap_5pos5neg_contrib': [],
        'top_shap_5pos5neg_ratio': [],
        'top_shap_5pos5neg_rank': [],
        'combined_3n3s_contrib': [],
        'combined_3n3s_ratio': [],
        'combined_3n3s_rank': [],
        'top_node_ratio5_contrib': [],
        'top_node_ratio5_ratio': [],
        'top_node_ratio5_rank': [],
        'top_shap_ratio5_contrib': [],
        'top_shap_ratio5_ratio': [],
        'top_shap_ratio5_rank': [],
        'combined_ratio3_contrib': [],
        'combined_ratio3_ratio': [],
        'combined_ratio3_rank': []
    }

    # Process each sample in this run
    for sample_idx in sample_indices:
        sample_X = X_selected.iloc[sample_idx].values.astype(np.float32)
        sample_to_explain = sample_X.reshape(1, -1)

        # Compute SHAP values for this run-sample combination
        def model_predict(data):
            tensor_data = torch.tensor(data, dtype=torch.float32)
            with torch.no_grad():
                outputs = model(tensor_data)
            return outputs.numpy()

        explainer = shap.KernelExplainer(model_predict, background)
        shap_values = explainer.shap_values(sample_to_explain, silent=True)[0]

        # Compute node contributions for this run-sample combination
        node_contrib, _ = compute_node_contributions(model, sample_X, 
                                                   selected_adjacency_mask, 
                                                   selected_features)

        # Create combined DataFrame for this run-sample
        combined_df = pd.DataFrame({
            'Feature': selected_features,
            'Average_SHAP': shap_values,
            'Node_Contribution': [node_contrib[f] for f in selected_features],
            'SHAP_Positive': (shap_values > 0).astype(int),
            'SHAP_Negative': (shap_values < 0).astype(int),
            # Fix: Convert list to numpy array before using astype()
            'Node_Positive': np.array([v > 0 for v in node_contrib.values()]).astype(int),
            'Node_Negative': np.array([v < 0 for v in node_contrib.values()]).astype(int)
        })

        # Calculate positive ratios
        combined_df['Node_Positive_Ratio'] = combined_df.apply(
            lambda x: x['Node_Positive'] / (x['Node_Positive'] + x['Node_Negative']) 
            if (x['Node_Positive'] + x['Node_Negative']) > 0 else 0.5, axis=1)
        
        combined_df['SHAP_Positive_Ratio'] = combined_df.apply(
            lambda x: x['SHAP_Positive'] / (x['SHAP_Positive'] + x['SHAP_Negative']) 
            if (x['SHAP_Positive'] + x['SHAP_Negative']) > 0 else 0.5, axis=1)

        # Get feature rankings for this run-sample
        node_rank = combined_df.sort_values('Node_Contribution', 
                                          ascending=False)['Feature'].tolist()
        shap_rank = combined_df.sort_values('Average_SHAP', 
                                          ascending=False)['Feature'].tolist()

        # Calculate correlations for different feature sets
        # 1. All features
        c, r, s = calculate_correlations(combined_df)
        run_correlations['all_features_contrib'].append(c)
        run_correlations['all_features_ratio'].append(r)
        run_correlations['all_features_rank'].append(s)

        # 2. Top 5 node pos + 5 node neg
        top_node = node_rank[:5] + node_rank[-5:]
        subset = combined_df[combined_df['Feature'].isin(top_node)]
        c, r, s = calculate_correlations(subset)
        run_correlations['top_node_5pos5neg_contrib'].append(c)
        run_correlations['top_node_5pos5neg_ratio'].append(r)
        run_correlations['top_node_5pos5neg_rank'].append(s)

        # 3. Top 5 SHAP pos + 5 SHAP neg
        top_shap = shap_rank[:5] + shap_rank[-5:]
        subset = combined_df[combined_df['Feature'].isin(top_shap)]
        c, r, s = calculate_correlations(subset)
        run_correlations['top_shap_5pos5neg_contrib'].append(c)
        run_correlations['top_shap_5pos5neg_ratio'].append(r)
        run_correlations['top_shap_5pos5neg_rank'].append(s)

        # 4. Combined 3n+3s
        combined_features = top_node[:3] + top_node[-3:] + top_shap[:3] + top_shap[-3:]
        subset = combined_df[combined_df['Feature'].isin(combined_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['combined_3n3s_contrib'].append(c)
        run_correlations['combined_3n3s_ratio'].append(r)
        run_correlations['combined_3n3s_rank'].append(s)

        # 5. Top node ratios (weighted by 5)
        node_ratio_features = combined_df.sort_values('Node_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                            combined_df.sort_values('Node_Positive_Ratio').head(5)['Feature'].tolist()
        subset = combined_df[combined_df['Feature'].isin(node_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['top_node_ratio5_contrib'].append(c)
        run_correlations['top_node_ratio5_ratio'].append(r)
        run_correlations['top_node_ratio5_rank'].append(s)

        # 6. Top SHAP ratios (weighted by 3)
        shap_ratio_features = combined_df.sort_values('SHAP_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                            combined_df.sort_values('SHAP_Positive_Ratio').head(5)['Feature'].tolist()
        subset = combined_df[combined_df['Feature'].isin(shap_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['top_shap_ratio5_contrib'].append(c)
        run_correlations['top_shap_ratio5_ratio'].append(r)
        run_correlations['top_shap_ratio5_rank'].append(s)

        # 7. Combined ratios
        combined_ratio_features = node_ratio_features[:3] + node_ratio_features[-3:] + shap_ratio_features[:3] + shap_ratio_features[-3:]
        subset = combined_df[combined_df['Feature'].isin(combined_ratio_features)]
        c, r, s = calculate_correlations(subset)
        # Changed from sample_results to run_correlations with append
        run_correlations['combined_ratio3_contrib'].append(c)
        run_correlations['combined_ratio3_ratio'].append(r)
        run_correlations['combined_ratio3_rank'].append(s)

    # Average correlations across samples for this run
    averaged_run = {k: np.nanmean(v) for k, v in run_correlations.items()}
    all_run_results.append(averaged_run)

# Create final summary DataFrame
summary_df = pd.DataFrame(all_run_results)
final_summary = pd.DataFrame()

for col in summary_df.columns:
    final_summary[col] = [
        summary_df[col].mean(),
        summary_df[col].std(),
        summary_df[col].median(),
        summary_df[col].quantile(0.25),
        summary_df[col].quantile(0.75)
    ]

final_summary.index = ['mean', 'std', 'median', '25%', '75%']
print(final_summary.transpose())

Processing runs: 100%|██████████████████████████████████████████████████████████████| 20/20 [2:28:15<00:00, 444.78s/it]

                               mean       std    median       25%       75%
all_features_contrib       0.011367  0.020792  0.010857 -0.006298  0.027117
all_features_ratio         0.069886  0.029366  0.064701  0.048714  0.089018
all_features_rank          0.030961  0.028123  0.031375  0.022972  0.047000
top_node_5pos5neg_contrib  0.001719  0.061708  0.001387 -0.037144  0.048624
top_node_5pos5neg_ratio    0.031831  0.058431  0.025060 -0.002494  0.077134
top_node_5pos5neg_rank     0.018917  0.060429  0.016496 -0.024900  0.076600
top_shap_5pos5neg_contrib  0.029474  0.059198  0.037462 -0.005000  0.067263
top_shap_5pos5neg_ratio    0.217580  0.077943  0.230752  0.159208  0.270507
top_shap_5pos5neg_rank     0.062752  0.070119  0.056182  0.020273  0.106606
combined_3n3s_contrib      0.003231  0.031708  0.007741 -0.024913  0.030233
combined_3n3s_ratio        0.142831  0.051077  0.132561  0.111993  0.178668
combined_3n3s_rank         0.029377  0.040483  0.026406  0.005804  0.058903
top_node_rat

In [14]:
import shap
from tqdm import tqdm
from scipy.stats import spearmanr

def calculate_jaccard(set1, set2):
    """Calculate Jaccard similarity between two sets."""
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union != 0 else 0

def calculate_rank_correlation(rank1, rank2):
    """Calculate Spearman correlation between two ranked lists."""
    common_items = list(set(rank1) & set(rank2))
    if len(common_items) < 2:
        return np.nan
    rank1_pos = [rank1.index(item) for item in common_items]
    rank2_pos = [rank2.index(item) for item in common_items]
    return spearmanr(rank1_pos, rank2_pos).correlation

def calculate_correlations(subset_df):
    """Calculate Pearson and Spearman correlations with error handling."""
    try:
        contrib_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='pearson').iloc[0,1]
    except:
        contrib_corr = np.nan
    try:
        ratio_corr = subset_df[['Node_Positive_Ratio', 'SHAP_Positive_Ratio']].corr(method='pearson').iloc[0,1]
    except:
        ratio_corr = np.nan
    try:
        rank_corr = subset_df[['Node_Contribution', 'Average_SHAP']].corr(method='spearman').iloc[0,1]
    except:
        rank_corr = np.nan
    return contrib_corr, ratio_corr, rank_corr

def compute_node_contributions(model, sample_X, adjacency_mask, selected_features, leaky_relu_negative_slope=0.01):
    # Extract model parameters
    W = model.W.detach().cpu().numpy()
    bias = model.bias.detach().cpu().numpy()
    batch_norm = model.batch_norm
    bn_weight = batch_norm.weight.detach().cpu().numpy()
    bn_bias = batch_norm.bias.detach().cpu().numpy()
    bn_running_mean = batch_norm.running_mean.detach().cpu().numpy()
    bn_running_var = batch_norm.running_var.detach().cpu().numpy()
    bn_eps = batch_norm.eps
    output_weights = model.output_weights.detach().cpu().numpy()
    
    x = sample_X
    num_features = len(selected_features)
    adjacency = adjacency_mask.numpy()

    # Precompute node inputs and activations
    node_inputs = np.zeros(num_features)
    for k in range(num_features):
        connected_i = np.where(adjacency[:, k] == 1)[0]
        node_inputs[k] = np.sum(W[connected_i, k] * x[connected_i] * x[k])

    # Process through batch norm and activation
    node_inputs_bn = (node_inputs - bn_running_mean) / np.sqrt(bn_running_var + bn_eps) * bn_weight + bn_bias
    node_inputs_bn += bias
    activations = np.where(node_inputs_bn > 0, node_inputs_bn, leaky_relu_negative_slope * node_inputs_bn)
    
    # Direct contributions (i->j->output)
    direct_contributions = activations * output_weights

    # Indirect contributions (j->k->output)
    indirect_contributions = np.zeros(num_features)
    for j in range(num_features):
        # Find all k that j connects to (outgoing)
        connected_ks = np.where(adjacency[j, :] == 1)[0]
        for k in connected_ks:
            # Calculate j's contribution to k's node input
            j_contribution = W[j, k] * x[j] * x[k]
            total_k_input = node_inputs[k]
            
            if total_k_input == 0:
                proportion = 0
            else:
                proportion = j_contribution / total_k_input
                
            # Attribute proportion of k's contribution back to j
            indirect_contributions[j] += proportion * direct_contributions[k]

    # Total contributions = direct + indirect
    total_contributions = direct_contributions + indirect_contributions

    # Normalize by sum of absolute values of all contributions in this sample
    sum_abs = np.sum(np.abs(total_contributions))
    if sum_abs > 0:
        total_contributions = total_contributions / sum_abs

    # Convert to dictionary
    node_contrib_dict = {
        feature: total_contributions[i]
        for i, feature in enumerate(selected_features)
    }
    
    return node_contrib_dict, np.sum(total_contributions)

# Configuration
num_runs = 20
num_samples = 100
np.random.seed(42)
sample_indices = np.random.choice(len(X_selected), size=num_samples, replace=False)
background = shap.sample(X_selected, 100, random_state=42).values.astype(np.float32)

# Initialize results storage
sample_correlations = []

for sample_idx in tqdm(sample_indices, desc="Processing samples"):
    all_shap = []
    all_node_contrib = []
    
    for run in range(1, num_runs + 1):
        # Load model for this run
        model_path = f"custom_network_trained_run_class123_{run}.pth"
        model = load_model(model_path, len(selected_features), selected_adjacency_mask)
        model.eval()
        
        sample_X = X_selected.iloc[sample_idx].values.astype(np.float32)
        sample_to_explain = sample_X.reshape(1, -1)
        
        # Compute SHAP values
        def model_predict(data):
            tensor_data = torch.tensor(data, dtype=torch.float32)
            with torch.no_grad():
                outputs = model(tensor_data)
            return outputs.numpy()
        
        explainer = shap.KernelExplainer(model_predict, background)
        shap_values = explainer.shap_values(sample_to_explain, silent=True)[0]
        
        # Compute node contributions
        node_contrib, _ = compute_node_contributions(model, sample_X, 
                                                   selected_adjacency_mask, 
                                                   selected_features)
        node_contrib_values = [node_contrib[f] for f in selected_features]
        
        all_shap.append(shap_values)
        all_node_contrib.append(node_contrib_values)
    
    # Convert to numpy arrays
    all_shap = np.array(all_shap)
    all_node_contrib = np.array(all_node_contrib)
    
    # Calculate average values
    avg_shap = np.mean(all_shap, axis=0)
    avg_node_contrib = np.mean(all_node_contrib, axis=0)
    
    # Calculate positive ratios across runs
    shap_pos_ratio = np.mean(all_shap > 0, axis=0)
    node_pos_ratio = np.mean(all_node_contrib > 0, axis=0)
    
    # Create combined DataFrame
    combined_df = pd.DataFrame({
        'Feature': selected_features,
        'Average_SHAP': avg_shap,
        'Node_Contribution': avg_node_contrib,
        'SHAP_Positive_Ratio': shap_pos_ratio,
        'Node_Positive_Ratio': node_pos_ratio
    })
    
    # Get feature rankings
    node_rank = combined_df.sort_values('Node_Contribution', ascending=False)['Feature'].tolist()
    shap_rank = combined_df.sort_values('Average_SHAP', ascending=False)['Feature'].tolist()
    
    # Calculate correlations for different feature sets
    sample_results = {}
    
    # 1. All features
    c, r, s = calculate_correlations(combined_df)
    sample_results['all_features_contrib'] = c
    sample_results['all_features_ratio'] = r
    sample_results['all_features_rank'] = s
    
    # 2. Top 5 node pos + 5 node neg
    top_node = node_rank[:5] + node_rank[-5:]
    subset = combined_df[combined_df['Feature'].isin(top_node)]
    c, r, s = calculate_correlations(subset)
    sample_results['top_node_5pos5neg_contrib'] = c
    sample_results['top_node_5pos5neg_ratio'] = r
    sample_results['top_node_5pos5neg_rank'] = s
    
    # 3. Top 5 SHAP pos + 5 SHAP neg
    top_shap = shap_rank[:5] + shap_rank[-5:]
    subset = combined_df[combined_df['Feature'].isin(top_shap)]
    c, r, s = calculate_correlations(subset)
    sample_results['top_shap_5pos5neg_contrib'] = c
    sample_results['top_shap_5pos5neg_ratio'] = r
    sample_results['top_shap_5pos5neg_rank'] = s
    
    # 4. Combined 3n+3s
    combined_features = top_node[:3] + top_node[-3:] + top_shap[:3] + top_shap[-3:]
    subset = combined_df[combined_df['Feature'].isin(combined_features)]
    c, r, s = calculate_correlations(subset)
    sample_results['combined_3n3s_contrib'] = c
    sample_results['combined_3n3s_ratio'] = r
    sample_results['combined_3n3s_rank'] = s
    
    # 5. Top node ratios
    node_ratio_features = combined_df.sort_values('Node_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                        combined_df.sort_values('Node_Positive_Ratio').head(5)['Feature'].tolist()
    subset = combined_df[combined_df['Feature'].isin(node_ratio_features)]
    c, r, s = calculate_correlations(subset)
    sample_results['top_node_ratio5_contrib'] = c
    sample_results['top_node_ratio5_ratio'] = r
    sample_results['top_node_ratio5_rank'] = s
    
    # 6. Top SHAP ratios
    shap_ratio_features = combined_df.sort_values('SHAP_Positive_Ratio', ascending=False).head(5)['Feature'].tolist() + \
                        combined_df.sort_values('SHAP_Positive_Ratio').head(5)['Feature'].tolist()
    subset = combined_df[combined_df['Feature'].isin(shap_ratio_features)]
    c, r, s = calculate_correlations(subset)
    sample_results['top_shap_ratio5_contrib'] = c
    sample_results['top_shap_ratio5_ratio'] = r
    sample_results['top_shap_ratio5_rank'] = s
    
    # 7. Combined ratios
    combined_ratio_features = node_ratio_features[:3] + node_ratio_features[-3:] + \
                            shap_ratio_features[:3] + shap_ratio_features[-3:]
    subset = combined_df[combined_df['Feature'].isin(combined_ratio_features)]
    c, r, s = calculate_correlations(subset)
    sample_results['combined_ratio3_contrib'] = c
    sample_results['combined_ratio3_ratio'] = r
    sample_results['combined_ratio3_rank'] = s
    
    sample_correlations.append(sample_results)

# Create final summary DataFrame
summary_df = pd.DataFrame(sample_correlations)
final_summary = pd.DataFrame()

for col in summary_df.columns:
    final_summary[col] = [
        summary_df[col].mean(),
        summary_df[col].std(),
        summary_df[col].median(),
        summary_df[col].quantile(0.25),
        summary_df[col].quantile(0.75)
    ]

final_summary.index = ['mean', 'std', 'median', '25%', '75%']
print(final_summary.transpose())

Processing samples: 100%|█████████████████████████████████████████████████████████| 100/100 [2:51:43<00:00, 103.04s/it]

                               mean       std    median       25%       75%
all_features_contrib       0.432595  0.171193  0.446508  0.326830  0.532401
all_features_ratio         0.216271  0.093869  0.223943  0.176756  0.283707
all_features_rank          0.239616  0.110369  0.239495  0.169102  0.327044
top_node_5pos5neg_contrib  0.720708  0.256692  0.796896  0.666446  0.883926
top_node_5pos5neg_ratio    0.726897  0.258784  0.790268  0.580535  0.924397
top_node_5pos5neg_rank     0.712970  0.266977  0.781818  0.648485  0.878788
top_shap_5pos5neg_contrib  0.682784  0.264769  0.762191  0.588971  0.866429
top_shap_5pos5neg_ratio    0.605745  0.286005  0.695992  0.395083  0.795978
top_shap_5pos5neg_rank     0.628242  0.261616  0.721212  0.466667  0.821212
combined_3n3s_contrib      0.581678  0.234065  0.636798  0.466039  0.739642
combined_3n3s_ratio        0.687773  0.249279  0.746682  0.597434  0.852730
combined_3n3s_rank         0.525199  0.196714  0.548485  0.440559  0.651136
top_node_rat

In [16]:
def analyze_consistent_weights(selected_features, selected_adjacency_mask, num_runs=20):
    """
    Analyze inter-feature weights (W[i,j]) with consistent signs across multiple runs.
    
    Args:
        selected_features (list): List of selected feature names
        selected_adjacency_mask (torch.Tensor): Adjacency mask for selected features
        num_runs (int): Number of model runs to analyze
        
    Returns:
        dict: Contains consistent pairs, counts, and total possible connections
    """
    # Convert adjacency mask to numpy array
    adjacency_np = selected_adjacency_mask.numpy()
    num_selected = len(selected_features)
    
    # Dictionary to store signs for each (i,j) connection across runs
    all_signs = {}

    # Analyze each model run
    for run in range(1, num_runs + 1):
        model_filename = f"custom_network_trained_run_class123_{run}.pth"
        if not os.path.exists(model_filename):
            print(f"Model file {model_filename} not found. Skipping Run {run}.")
            continue

        # Load model and extract weights
        model = CustomNetwork(num_features=num_selected, 
                             adjacency_mask=selected_adjacency_mask)
        model.load_state_dict(torch.load(model_filename, map_location='cpu'))
        model.eval()
        
        with torch.no_grad():
            W = model.W.cpu().numpy()

        # Record signs for active connections
        for i in range(num_selected):
            for j in range(num_selected):
                if adjacency_np[i, j] == 1:  # Only consider existing connections
                    sign = np.sign(W[i, j])
                    key = (i, j)
                    if key not in all_signs:
                        all_signs[key] = []
                    all_signs[key].append(sign)

    # Identify consistently signed weights
    consistent_pairs = []
    for (i, j), signs in all_signs.items():
        if len(signs) != num_runs:
            continue  # Skip incomplete data
            
        unique_signs = set(signs)
        if len(unique_signs) == 1 and 0 not in unique_signs:
            consistent_pairs.append({
                'Source': selected_features[i],
                'Target': selected_features[j],
                'Sign': 'Positive' if unique_signs.pop() > 0 else 'Negative'
            })

    # Calculate statistics
    total_connections = int(adjacency_np.sum())
    num_consistent = len(consistent_pairs)
    
    # Print results
    print(f"\nConsistent Weights: {num_consistent}/{total_connections} connections")
    print("=============================================")
    if consistent_pairs:
        df = pd.DataFrame(consistent_pairs)
        print(df.to_string(index=False))
    else:
        print("No weights with consistent signs across all runs")
    
    return {
        'consistent_pairs': consistent_pairs,
        'num_consistent': num_consistent,
        'total_connections': total_connections
    }

# Execute the analysis
weight_analysis = analyze_consistent_weights(
    selected_features=selected_features,
    selected_adjacency_mask=selected_adjacency_mask,
    num_runs=20
)


Consistent Weights: 164/27225 connections
                              Source                               Target     Sign
                            rs591058                            rs1800972 Negative
                           rs2104772                              rs12722 Negative
                           rs1144393                            rs4654760 Positive
                           rs1144393                            rs1800795 Positive
                           rs1144393                       Flight_time_12 Positive
                           rs1144393             hip_adduction_peak_angle Positive
                           rs1144393      resistance_training_past_season Positive
                           rs1544410                             rs911263 Negative
                           rs1544410                        leg_lean_mass Negative
                           rs2237352                           rs10992075 Negative
                           rs2237352        